# Testing the documentation extraction pipeline

This notebook imports [`dataset/buildDataset/build.py`](dataset/buildDataset/build.py)
directly and calls its actual functions (`find_documentation_header`,
`extract_documentation`, `strip_comments`, `tokenize`, `calculate_entropy`,
`doc_redundancy`, `doc_code_overlap`) on test strings, so the exact code used
for the paper's mining pipeline is what gets exercised here -- no hand-typed
copies.

The only function defined locally is `clean_comments`, ported from
[`dataset/buildDataset/combine.ipynb`](dataset/buildDataset/combine.ipynb) --
that one can't be imported since it lives in a notebook, not a `.py` module.

## 1. Import `build.py` directly

In [1]:
import os
import sys
import re
import textwrap

import numpy as np
import textstat

# build.py raises ValueError at import time unless a GITHUB_TOKEN_* env var is set.
# That check happens before any network call, so a dummy value satisfies it without
# contacting GitHub. None of the functions used below (tokenize, calculate_entropy,
# doc_redundancy, doc_code_overlap, strip_comments, find_documentation_header,
# extract_documentation) touch the network, git, or the filesystem beyond this.
os.environ.setdefault("GITHUB_TOKEN_1", "dummy-token-for-local-testing")

BUILD_DIR = os.path.join(os.getcwd(), "dataset", "buildDataset")
sys.path.insert(0, BUILD_DIR)

import build

print("Imported build.py successfully -- using its functions directly below.")


GitHub token loaded successfully.
Imported build.py successfully -- using its functions directly below.


## 2. `clean_comments` (from `combine.ipynb`) -- the one function that can't be imported

In [2]:
def clean_comments(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. Extract Block/Docstring Content
    blocks = re.findall(r'/\*+([\s\S]*?)\*/', text)
    docstrings = re.findall(r'["\']{3}([\s\S]*?)["\']{3}', text)

    # 2. Extract Single-Line
    inline = re.findall(r'(?:#|///|//)\s*(.*?)\s*(?=#|///|//|$)', text)

    combined = blocks + docstrings + inline
    cleaned_comments = []

    for item in combined:
        # Remove the leading '*' from each line (common in JSDoc/C-style)
        clean_item = re.sub(r'^\s*\* ?', '', item, flags=re.MULTILINE)

        # Collapse newlines and tabs into a single space
        clean_item = ' '.join(clean_item.split())

        if clean_item.strip():
            cleaned_comments.append(clean_item.strip())

    return ' '.join(cleaned_comments)


## 3. Test strings

In [3]:
test_code_py = '''
"""ArithmeticError"""
"""
top test
"""
def _generate_semantic_suggestions(kind: RelationshipKind, source_type: str, target_type: str) -> List[str]:
    """Generate suggestions based on semantic analysis of relationship types.
    blat bang
    """
    suggestions = []
    """
    shmayes da slim
    """
    a = """ArithmeticError fuck shat"""
    # Categorize relationships by semantic meaning
    governance_relations = {
        RelationshipKind.GOVERNS, RelationshipKind.REGULATES, RelationshipKind.MANDATES,
        RelationshipKind.AUTHORIZES, RelationshipKind.ENFORCES, RelationshipKind.DELEGATES,
        RelationshipKind.LICENSES, RelationshipKind.CERTIFIES, RelationshipKind.SANCTIONS
    }

    resource_flow_relations = {
        RelationshipKind.FUNDS, RelationshipKind.PAYS, RelationshipKind.ALLOCATES,
        RelationshipKind.TRANSFERS, RelationshipKind.SUPPLIES, RelationshipKind.PRODUCES,
        RelationshipKind.DISTRIBUTES, RelationshipKind.CONVERTS, RelationshipKind.EXCHANGES_WITH
    }

    knowledge_relations = {
        RelationshipKind.INFORMS, RelationshipKind.EDUCATES, RelationshipKind.ADVISES,
        RelationshipKind.RESEARCHES, RelationshipKind.ANALYZES, RelationshipKind.COMMUNICATES_WITH,
        RelationshipKind.DOCUMENTS, RelationshipKind.MEASURES
    }

    collaborative_relations = {
        RelationshipKind.COLLABORATES_WITH, RelationshipKind.COORDINATES_WITH,
        RelationshipKind.SUPPORTS, RelationshipKind.ALLIES_WITH, RelationshipKind.FACILITATES,
        RelationshipKind.PARTICIPATES_IN, RelationshipKind.ORGANIZES
    }

    # Provide semantic category guidance
    if kind in governance_relations:
        if source_type not in ['Actor', 'Institution', 'Policy']:
            suggestions.append(f""Governance relationships like {kind.name} typically require Actor, Institution, or Policy as source"")
        if target_type in ['Actor', 'Institution', 'Policy', 'Resource']:
            suggestions.append(f""Consider {kind.name} with governable entities: Actor, Institution, Policy, or Resource"")

    elif kind in resource_flow_relations:
        if source_type not in ['Actor', 'Institution', 'Process', 'PolicyInstrument']:
            suggestions.append(f""Resource flow relationships like {kind.name} typically require entities capable of resource handling"")
        if target_type not in ['Actor', 'Resource', 'Flow', 'ValueFlow']:
            suggestions.append(f""Consider {kind.name} targeting resource-receiving entities: Actor, Resource, Flow, or ValueFlow"")

    elif kind in knowledge_relations:
        if source_type not in ['Actor', 'Institution', 'TechnologySystem']:
            suggestions.append(f""Knowledge relationships like {kind.name} typically require information-capable entities"")
        if target_type not in ['Actor', 'Institution', 'Resource', 'TechnologySystem']:
            suggestions.append(f""Consider {kind.name} with information-receiving entities"")

    elif kind in collaborative_relations:
        if source_type not in ['Actor', 'Institution']:
            suggestions.append(f""Collaborative relationships like {kind.name} typically require social entities like Actor or Institution"")
        if target_type not in ['Actor', 'Institution', 'Process']:
            suggestions.append(f""Consider {kind.name} with collaborative entities: Actor, Institution, or Process"")

    return suggestions
'''

test_code_js = '''/**
 * Computes the factorial of n using recursion.
 * @param {number} n - non-negative integer
 * @returns {number} factorial of n
 */
function factorial(n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}
'''
# ---------------------------------------------------------------- C# (.cs)
test_code_cs = '''/// <summary>
/// Computes the factorial of n using recursion.
/// </summary>
/// <param name="n">non-negative integer</param>
/// <returns>factorial of n</returns>
public static long Factorial(int n)
{
    // base case
    if (n <= 1) return 1;
    return n * Factorial(n - 1); // recursive case
}
'''
 
# ------------------------------------------------------------- C / C++ (.c, .cpp, .h, ...)
test_code_c = '''/**
 * Computes the factorial of n using recursion.
 * @param n non-negative integer
 * @return factorial of n
 */
long factorial(int n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}
'''
 
# ---------------------------------------------------------------- Go (.go)
test_code_go = '''// Factorial computes the factorial of n using recursion.
// n must be a non-negative integer.
func Factorial(n int) int {
    // base case
    if n <= 1 {
        return 1
    }
    return n * Factorial(n-1) // recursive case
}
'''
 
# -------------------------------------------------------------- Java (.java)
test_code_java = '''/**
 * Computes the factorial of n using recursion.
 * @param n non-negative integer
 * @return factorial of n
 */
public static long factorial(int n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}
'''
 
# ------------------------------------------------------ JavaScript (.js, .jsx)
test_code_js = '''/**
 * Computes the factorial of n using recursion.
 * @param {number} n - non-negative integer
 * @returns {number} factorial of n
 */
function factorial(n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}
'''
 
# ---------------------------------------------------- Kotlin (.kt, .kts)
test_code_kt = '''/**
 * Computes the factorial of n using recursion.
 * @param n non-negative integer
 * @return factorial of n
 */
fun factorial(n: Int): Long {
    // base case
    if (n <= 1) return 1
    return n * factorial(n - 1) // recursive case
}
'''
 
# ---------------------------------------------------------------- PHP (.php)
test_code_php = '''/**
 * Computes the factorial of n using recursion.
 * @param int $n non-negative integer
 * @return int factorial of n
 */
function factorial($n) {
    # base case
    if ($n <= 1) return 1;
    return $n * factorial($n - 1); // recursive case
}
'''
 
# ------------------------------------------------------------- Python (.py)
test_code_py = '''def factorial(n):
    """
    Computes the factorial of n using recursion.
    :param n: non-negative integer
    :return: factorial of n
    """
    # base case
    if n <= 1:
        return 1
    return n * factorial(n - 1)  # recursive case
'''
 
# ---------------------------------------------------------- Scala (.scala)
test_code_scala = '''/**
 * Computes the factorial of n using recursion.
 * @param n non-negative integer
 * @return factorial of n
 */
def factorial(n: Int): Long = {
    // base case
    if (n <= 1) 1
    else n * factorial(n - 1) // recursive case
}
'''
 
# ---------------------------------------------------------- Swift (.swift)
test_code_swift = '''/// Computes the factorial of n using recursion.
/// - Parameter n: non-negative integer
/// - Returns: factorial of n
func factorial(_ n: Int) -> Int {
    // base case
    if n <= 1 { return 1 }
    return n * factorial(n - 1) // recursive case
}
'''
 
print(test_code_py)
print("========================================")
print(test_code_js)


def factorial(n):
    """
    Computes the factorial of n using recursion.
    :param n: non-negative integer
    :return: factorial of n
    """
    # base case
    if n <= 1:
        return 1
    return n * factorial(n - 1)  # recursive case

/**
 * Computes the factorial of n using recursion.
 * @param {number} n - non-negative integer
 * @returns {number} factorial of n
 */
function factorial(n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}



## 4. Run `build.find_documentation_header` + `build.extract_documentation` on every language sample, and verify each one's documentation was captured correctly

In [4]:
factorial_samples = {
    "Python":     (test_code_py,    1, ".py"),
    "JavaScript": (test_code_js,    6, ".js"),
    "C#":         (test_code_cs,    6, ".cs"),
    "C/C++":      (test_code_c,     6, ".c"),
    "Go":         (test_code_go,    3, ".go"),
    "Java":       (test_code_java,  6, ".java"),
    "Kotlin":     (test_code_kt,    6, ".kt"),
    "PHP":        (test_code_php,   6, ".php"),
    "Scala":      (test_code_scala, 6, ".scala"),
    "Swift":      (test_code_swift, 4, ".swift"),
}

must_contain = ["factorial of n using recursion", "base case", "recursive case"]

extraction_results = {}
all_passed = True

for lang, (code, func_line, ext) in factorial_samples.items():
    lines = code.splitlines()
    adj_start = build.find_documentation_header(lines, func_line)
    doc_list, _ = build.extract_documentation(lines, adj_start, len(lines), ext)
    doc_text = " ".join(doc_list)

    raw_code = "\n".join(lines[adj_start - 1:len(lines)])
    code_text = textwrap.dedent(raw_code)
    code_text_no_doc = build.strip_comments(code_text)

    missing = [s for s in must_contain if s not in doc_text]
    ok = not missing
    all_passed &= ok

    extraction_results[lang] = {
        "doc_text": doc_text,
        "code_text_no_doc": code_text_no_doc,
    }

    print(f"=== {lang} ({ext}) -- {'PASS' if ok else 'FAIL'} ===")
    print(f"function line: {func_line} | adjusted doc start: {adj_start}")
    print("doc_text:", repr(doc_text))
    if missing:
        print(f"  MISSING expected content: {missing}")
    print()

print("ALL LANGUAGES CAPTURED THEIR DOCUMENTATION CORRECTLY" if all_passed else "SOME LANGUAGES FAILED")


=== Python (.py) -- PASS ===
function line: 1 | adjusted doc start: 1
doc_text: '"""\n    Computes the factorial of n using recursion.\n    :param n: non-negative integer\n    :return: factorial of n\n    """ # base case # recursive case'

=== JavaScript (.js) -- PASS ===
function line: 6 | adjusted doc start: 1
doc_text: '/**\n * Computes the factorial of n using recursion.\n * @param {number} n - non-negative integer\n * @returns {number} factorial of n\n */ // base case // recursive case'

=== C# (.cs) -- PASS ===
function line: 6 | adjusted doc start: 1
doc_text: '/// <summary> /// Computes the factorial of n using recursion. /// </summary> /// <param name="n">non-negative integer</param> /// <returns>factorial of n</returns> // base case // recursive case'

=== C/C++ (.c) -- PASS ===
function line: 6 | adjusted doc start: 1
doc_text: '/**\n * Computes the factorial of n using recursion.\n * @param n non-negative integer\n * @return factorial of n\n */ // base case // recursive cas

## 5. Apply `clean_comments` (from `combine.ipynb`) to each language's extracted text

In [5]:
for lang, result in extraction_results.items():
    cleaned = clean_comments(result["doc_text"])
    result["doc_cleaned"] = cleaned
    print(f"--- {lang} -- cleaned doc text ---")
    print(cleaned)
    print()


--- Python -- cleaned doc text ---
Computes the factorial of n using recursion. :param n: non-negative integer :return: factorial of n base case recursive case

--- JavaScript -- cleaned doc text ---
Computes the factorial of n using recursion. @param {number} n - non-negative integer @returns {number} factorial of n base case recursive case

--- C# -- cleaned doc text ---
<summary> Computes the factorial of n using recursion. </summary> <param name="n">non-negative integer</param> <returns>factorial of n</returns> base case recursive case

--- C/C++ -- cleaned doc text ---
Computes the factorial of n using recursion. @param n non-negative integer @return factorial of n base case recursive case

--- Go -- cleaned doc text ---
Factorial computes the factorial of n using recursion. n must be a non-negative integer. base case recursive case

--- Java -- cleaned doc text ---
Computes the factorial of n using recursion. @param n non-negative integer @return factorial of n base case recursiv

## 6. Compute the documentation metrics for each language using `build.py`'s own functions

In [6]:
import pandas as pd


def report_metrics(label, doc_text_clean, code_text_no_doc):
    entropy = round(build.calculate_entropy(doc_text_clean), 4) if doc_text_clean else np.nan
    redundancy = round(build.doc_redundancy(doc_text_clean), 4) if doc_text_clean else np.nan
    overlap = round(build.doc_code_overlap(doc_text_clean, code_text_no_doc), 4) if doc_text_clean else np.nan
    readability = textstat.flesch_reading_ease(doc_text_clean) if doc_text_clean else np.nan
    return entropy, redundancy, overlap, readability


rows = []
for lang, result in extraction_results.items():
    entropy, redundancy, overlap, readability = report_metrics(
        lang, result["doc_cleaned"], result["code_text_no_doc"]
    )
    rows.append({
        "language": lang,
        "doc_entropy": entropy,
        "doc_redundancy": redundancy,
        "doc_code_overlap": overlap,
        "doc_readability": readability,
    })

metrics_df = pd.DataFrame(rows).set_index("language")
metrics_df


,doc_entropy,doc_redundancy,doc_code_overlap,doc_readability
language,,,,
Python,3.7842,0.2500,0.8000,32.445132
JavaScript,3.8797,0.2727,0.1250,31.006071
C#,3.9737,0.3200,0.1176,-3.175921
C/C++,3.7842,0.2500,0.2000,32.445132
Go,3.9321,0.1579,0.1250,36.245000
Java,3.7842,0.2500,0.2000,32.445132
Kotlin,3.7842,0.2500,0.2000,32.445132
PHP,3.8797,0.2727,0.1875,39.063214
Scala,3.7842,0.2500,0.1333,32.445132


## 8. Per-language testing

`build.py` supports 11 languages (`SUPPORTED_EXTENSIONS`): C, C#, C++, Go,
Java, JavaScript, Kotlin, PHP, Python, Scala, and Swift. One representative
snippet per language is defined below, each using that language's normal
doc-comment convention, and run through the *actual* `build.find_documentation_header`
+ `build.extract_documentation` (imported directly, same as above).

In [18]:
language_samples = {
    # lang: (code, function_line, file_extension, must_contain, must_not_contain)
    "Python (real docstring)": ('''def foo(x):
    """
    Multi-line docstring.

    Returns something.
    """
    # inline comment
    return x
''', 1, ".py", ["Multi-line docstring", "inline comment"], []),

    "Python (docstring + later string literal)": ('''def foo(x):
    """
    Real docstring here.
    """
    a = """ArithmeticError fuck shat"""
    return a
''', 1, ".py", ["Real docstring here"], ["ArithmeticError"]),

    "Python (assigned string, NOT a docstring)": ('''def foo(x):
    a = """ArithmeticError fuck shat"""
    return a
''', 1, ".py", [], ["ArithmeticError"]),

    "JavaScript": ('''/**
 * JSDoc summary.
 * @param {number} x
 */
function foo(x) {
    // inline comment
    return x;
}
''', 4, ".js", ["JSDoc summary", "inline comment"], []),

    "Java (Javadoc)": ('''/**
 * Javadoc summary.
 * @param x input value
 */
public int foo(int x) {
    // inline comment
    return x;
}
''', 4, ".java", ["Javadoc summary", "inline comment"], []),

    "Java (text block, NOT a docstring)": ('''public String foo(int x) {
    // inline comment
    String html = """
        <html>
        </html>
        """;
    return html;
}
''', 1, ".java", ["inline comment"], ["<html>"]),

    "C": ('''/**
 * Doc comment.
 */
int foo(int x) {
    // inline comment
    return x;
}
''', 4, ".c", ["Doc comment", "inline comment"], []),

    "C++": ('''/**
 * Doc comment.
 */
int foo(int x) {
    // inline comment
    return x;
}
''', 4, ".cpp", ["Doc comment", "inline comment"], []),

    "C#": ('''/// <summary>
/// XML doc summary.
/// </summary>
public int Foo(int x) {
    // inline comment
    string raw = """
    {
      "key": "value"
    }
    """;
    return x;
}
''', 4, ".cs", ["XML doc summary", "inline comment"], ["key", "value"]),

    "Go": ('''// Foo returns x unchanged.
func Foo(x int) int {
    // inline comment
    return x
}
''', 2, ".go", ["Foo returns x unchanged", "inline comment"], []),

    "Kotlin (KDoc)": ('''/**
 * KDoc summary.
 */
fun foo(x: Int): Int {
    // inline comment
    return x
}
''', 4, ".kt", ["KDoc summary", "inline comment"], []),

    "Kotlin (raw string, NOT documentation)": ('''fun foo(x: Int): String {
    // inline comment
    val raw = """
        not documentation
        """
    return raw
}
''', 1, ".kt", ["inline comment"], ["not documentation"]),

    "PHP": ('''/**
 * phpDoc summary.
 * @param int $x
 */
function foo($x) {
    // inline comment
    return $x;
}
''', 4, ".php", ["phpDoc summary", "inline comment"], []),

    "Scala": ('''/**
 * Scaladoc summary.
 */
def foo(x: Int): Int = {
    // inline comment
    val raw = """
    not documentation, just a raw string
    """
    x
}
''', 4, ".scala", ["Scaladoc summary", "inline comment"], ["not documentation"]),

    "Swift": ('''/// Doc comment summary.
func foo(x: Int) -> Int {
    // inline comment
    // bggg
    let raw = """
    not documentation, just a raw string
    """
    return x
}
''', 2, ".swift", ["Doc comment summary", "inline comment"], ["not documentation"]),
}

all_passed = True
for lang, (code, func_line, ext, must_contain, must_not_contain) in language_samples.items():
    lines = code.splitlines()
    adj_start = build.find_documentation_header(lines, func_line)
    doc_list, _ = build.extract_documentation(lines, adj_start, len(lines), ext)
    doc_text = " ".join(doc_list)

    missing = [s for s in must_contain if s not in doc_text]
    leaked = [s for s in must_not_contain if s in doc_text]
    ok = not missing and not leaked
    all_passed &= ok

    print(f"=== {lang} ({ext}) -- {'PASS' if ok else 'FAIL'} ===")
    print(f"function line: {func_line} | adjusted doc start: {adj_start}")
    print("doc_text:", repr(doc_text))
    if missing:
        print(f"  MISSING expected content: {missing}")
    if leaked:
        print(f"  LEAKED non-doc content that should have been excluded: {leaked}")
    print()

print("ALL LANGUAGE TESTS PASSED" if all_passed else "SOME LANGUAGE TESTS FAILED")


=== Python (real docstring) (.py) -- PASS ===
function line: 1 | adjusted doc start: 1
doc_text: '"""\n    Multi-line docstring.\n\n    Returns something.\n    """ # inline comment'

=== Python (docstring + later string literal) (.py) -- PASS ===
function line: 1 | adjusted doc start: 1
doc_text: '"""\n    Real docstring here.\n    """'

=== Python (assigned string, NOT a docstring) (.py) -- PASS ===
function line: 1 | adjusted doc start: 1
doc_text: ''

=== JavaScript (.js) -- PASS ===
function line: 4 | adjusted doc start: 1
doc_text: '/**\n * JSDoc summary.\n * @param {number} x\n */ // inline comment'

=== Java (Javadoc) (.java) -- PASS ===
function line: 4 | adjusted doc start: 1
doc_text: '/**\n * Javadoc summary.\n * @param x input value\n */ // inline comment'

=== Java (text block, NOT a docstring) (.java) -- PASS ===
function line: 1 | adjusted doc start: 1
doc_text: '// inline comment'

=== C (.c) -- PASS ===
function line: 4 | adjusted doc start: 1
doc_text: '/**\n * Doc co

## 9. How many functions in the real dataset does this fix change?

Loads the actual mined dataset (`dataset/data/final_dataset_new_corrected.csv`,
13,809 functions, all 11 languages -- the same file behind the paper's
tables) and recomputes `doc_text` for every row using both the pre-fix and
post-fix `extract_documentation`, directly from the stored `function` column
(no repo re-cloning needed, since `function` already contains the full
source block the extraction runs over).

In [8]:
import pandas as pd

# Reconstruct the exact PRE-FIX (buggy) version for comparison
def old_extract_documentation(lines, start, end):
    raw_block = "\n".join(lines[start-1:end])
    extracted_docs = []
    c_blocks = re.findall(r'/\*.*?\*/', raw_block, flags=re.DOTALL)
    extracted_docs.extend([b.strip() for b in c_blocks])
    for line in raw_block.splitlines():
        line = line.strip()
        py_doc = re.findall(r'(""".*?"""|' + "'''.*?'''" + r'|"""[\s\S]*?"""|' + "'''[\\s\\S]*?''')", line)
        if py_doc:
            extracted_docs.extend(py_doc)
            continue
        comment_match = re.search(r'(//|#)(.*)$', line)
        if comment_match:
            extracted_docs.append(comment_match.group(0).strip())
    return extracted_docs, lines

df = pd.read_csv("dataset/data/final_dataset_new_corrected.csv")

ext_to_lang = {
    "cs": "C#", "c": "C", "cpp": "C++", "cc": "C++", "h": "C", "hpp": "C++",
    "cxx": "C++", "hxx": "C++", "go": "Go", "java": "Java", "js": "JavaScript",
    "jsx": "JavaScript", "kt": "Kotlin", "kts": "Kotlin", "php": "PHP",
    "py": "Python", "scala": "Scala", "swift": "Swift",
}
df["ext"] = df["file_path"].apply(lambda p: str(p).rsplit(".", 1)[-1].lower() if isinstance(p, str) and "." in p else "unknown")
df["language"] = df["ext"].map(ext_to_lang).fillna(df["ext"])

def recompute(func_text, ext):
    if not isinstance(func_text, str) or not func_text.strip():
        return "", ""
    lines = func_text.splitlines()
    old_doc, _ = old_extract_documentation(lines, 1, len(lines))
    new_doc, _ = build.extract_documentation(lines, 1, len(lines), "." + ext if ext else "")
    return " ".join(old_doc), " ".join(new_doc)

recomputed = df.apply(lambda row: recompute(row["function"], row["ext"]), axis=1)
df["old_doc_text"] = recomputed.apply(lambda t: t[0])
df["new_doc_text"] = recomputed.apply(lambda t: t[1])
df["changed"] = df["old_doc_text"] != df["new_doc_text"]

print(f"Total functions whose doc_text changes under the fix: {df['changed'].sum()} / {len(df)} ({100*df['changed'].mean():.2f}%)")
print()
print("Breakdown by language:")
summary = df.groupby("language").agg(total=("changed", "size"), changed=("changed", "sum"))
summary["pct"] = (100 * summary["changed"] / summary["total"]).round(2)
print(summary.sort_values("changed", ascending=False))


Total functions whose doc_text changes under the fix: 1002 / 13809 (7.26%)

Breakdown by language:
            total  changed    pct
language                         
Python       4211      943  22.39
JavaScript   4392       37   0.84
C++           814        8   0.98
PHP           595        7   1.18
Java         1481        6   0.41
Scala          46        1   2.17
C             515        0   0.00
C#            712        0   0.00
Go            723        0   0.00
Kotlin         84        0   0.00
Swift         236        0   0.00


### Note: not every changed row is a pure bug fix

Inspecting a few non-Python diffs shows two different things happening:

1. **Python (~976 rows) and a handful of JS/C++/Java/PHP rows (~57)**: genuine
   fixes -- either the multi-line docstring is now captured at all (Python),
   or a duplicate `//`/`#` fragment that the old code double-counted from
   *inside* an already-captured `/* */` block (e.g. a URL like
   `https://lmms.io` inside a header comment, matched a second time as a
   spurious `//lmms.io` "comment") is no longer duplicated.
2. **A small number of Scala/Swift/C# rows**: the broadened triple-quote
   regex now also matches ordinary **raw/multi-line string literals** that
   have nothing to do with documentation -- Swift and C# both support
   `"""..."""` multi-line string literals as a language feature (not just
   Python-style docstrings), and Scala supports triple-quoted raw strings
   too. These get mistakenly captured as "documentation" by the widened
   regex. This is a new, narrow false-positive introduced by the fix, on top
   of the intended Python fix -- worth a follow-up refinement (e.g. only
   treating a triple-quoted block as a docstring when it is the first
   statement immediately after the function signature) if it matters for
   your results.